# Part 1 — Acquisition and Storage
**BFT65B · Student ID 11230945**

This notebook implements the first stage of the bank-failure early-warning pipeline. Run the cells in order. Its outputs establish a reproducible source snapshot and compare storage layouts. Financial observations are neither cleaned nor used for modelling here.

**Excellent-level target (15 points):** complete and verified data; an efficient format, schema, partitioning and compression design justified by measurements. Required scope: all banks, **1984Q1–2026Q2 (170 quarters)**, complete Failures and Institutions, and explicitly selected fields. The grading rubric requires evidence rather than a particular compression ratio or speedup.

Evidence chain: **source definitions → smoke download → complete acquisition → independent audit → identical storage candidates → equivalence checks → repeated measurements → Part 2 handoff**.

See README for step-by-step GitHub/Colab operations, attribution, measurement boundaries and submission checks. Outputs supplied with this notebook are from the recorded verification environment; rerun in Colab to obtain measurements for your own runtime.

## 1. Prepare the repository and pinned environment
Use a fresh Python 3 CPU runtime. This cell clones the public repository in Colab and installs the supplied requirements. If Colab asks for a session restart, restart and rerun this cell before importing analysis libraries. Outside Colab, activate the documented virtual environment first.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        pass

REPOSITORY = 'https://github.com/NguyenDuong0508/11230945_BFT65B_Final_BigData.git'
if IN_COLAB:
    REPO_ROOT = Path('/content/11230945_BFT65B_Final_BigData')
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', REPOSITORY, str(REPO_ROOT)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')], check=True)
else:
    REPO_ROOT = Path(os.environ.get('BFT_REPO_ROOT', Path.cwd())).resolve()
    if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
        REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'src/acquisition.py').is_file(), 'Open this notebook from the supplied repository.'
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Repository:', REPO_ROOT)
print('Python:', sys.version.split()[0])
print('Execution environment:', 'Google Colab' if IN_COLAB else 'Linux verification/local runtime')

Repository: /workspace/work/part1
Python: 3.12.14
Execution environment: Linux verification/local runtime


## 2. Separate persistent source data from temporary benchmark files
Google Drive stores the source snapshot and checkpoints. Local runtime disk stores the benchmark layouts so Drive latency does not enter the format comparison. No bank-level data belong in GitHub. The full and smoke datasets use separate directories.

In [2]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_BASE = Path('/content/drive/MyDrive/BFT65B/part1_data')
    BENCH_ROOT = Path('/content/bft65b_benchmark')
else:
    DATA_BASE = Path(os.environ.get('BFT_DATA_BASE', REPO_ROOT / 'data'))
    BENCH_ROOT = Path(os.environ.get('BFT_BENCH_ROOT', REPO_ROOT / 'benchmark_work'))
FULL_ROOT = DATA_BASE / 'full'
SMOKE_ROOT = DATA_BASE / 'smoke_2023'
EVIDENCE = REPO_ROOT / 'evidence'
for folder in [DATA_BASE, BENCH_ROOT, EVIDENCE]:
    folder.mkdir(parents=True, exist_ok=True)
SEED = 42
START, END = '1984-03-31', '2026-06-30'
print('Persistent snapshot:', FULL_ROOT)
print('Temporary benchmark:', BENCH_ROOT)

Persistent snapshot: /workspace/work/execution_data/full
Temporary benchmark: /workspace/work/benchmark


## 3. Record the scope, schema and official field definitions

| Group | Selected fields | Purpose for subsequent stages |
|---|---|---|
| Identity | CERT, REPDTE, STNAME | Bank–quarter key, reporting time and geographic context |
| Balance sheet and funding | ASSET, DEP, EQ, BRO | Size, deposits, equity and brokered funding |
| Liquidity and securities | CHBAL, SC | Cash and securities exposures |
| Lending and credit quality | LNLSGR, LNATRES, NCLNLS | Gross loans, loss allowances and noncurrent loans |
| Earnings | NETINC, ROA, ROE, NIMY | Income and reported profitability ratios |
| Regulatory capital | RBC1AAJ, RBCRWAJ | Reported leverage and total risk-based capital ratios |
| Failure/assistance history | CERT, NAME, CITYST, FAILDATE, FAILYR, COST, QBFASSET, QBFDEP, RESTYPE | Event interpretation and potential cost analysis; not historical predictors |
| Institutions | CERT, NAME, STNAME, BKCLASS, ACTIVE, ESTYMD, ENDEFYMD, CHANGEC1, INSDATE | Identity, coverage and exit interpretation; current status is not a point-in-time feature |

The selection preserves a manageable set of economically relevant raw fields. It is not a fitted feature-selection result. Amounts and ratios retain source units; check definitions and reporting conventions before engineering ratios or quarterly flows. The Failures endpoint includes assisted institutions: retain RESTYPE for Part 2 rather than calling every row an eligible failure.

Storage uses Int64 for CERT/REPDTE/FAILYR, Float64 for financial measurements and strings for source names, auxiliary dates and categorical codes. Nulls stay null. Failures also retains the API-returned ID as string SOURCE_ID: this source key remains usable when CERT is missing. Such records are preserved for Part 2, and their counts appear in the audit. Financials are checkpointed by quarter inside year directories, compressed with Zstandard level 3. The alternative compact yearly layout is tested later.

In [3]:
import json
import platform
import importlib.metadata
import pandas as pd
from IPython.display import display
from src.acquisition import Client, FIELDS, quarters, schema, archive_dictionaries, acquire, audit, save_json, digest

assert len(quarters(START, END)) == 170
versions = {name: importlib.metadata.version(name) for name in ['numpy', 'pandas', 'pyarrow', 'polars', 'requests']}
print('Versions:', versions)
definitions = archive_dictionaries(FULL_ROOT, Client())
field_rows = []
for endpoint, definition in definitions.items():
    for field, details in definition['fields'].items():
        field_rows.append({'table': endpoint, 'field': field, 'storage_type': str(schema(endpoint).field(field).type),
                           'official_title': details.get('title', ''), 'definition_available': bool(details.get('description'))})
display(pd.DataFrame(field_rows))
print('Full definitions and source hashes are stored under metadata/selected_fields.json.')

Versions: {'numpy': '2.1.3', 'pandas': '2.2.3', 'pyarrow': '19.0.1', 'polars': '1.26.0', 'requests': '2.32.3'}


,table,field,storage_type,official_title,definition_available
0,financials,CERT,int64,FDIC Certificate #,True
1,financials,REPDTE,int64,Report Date,True
2,financials,STNAME,string,STATE NAME,False
3,financials,ASSET,double,Total assets,True
4,financials,DEP,double,Total deposits,True
5,financials,EQ,double,Equity capital,True
6,financials,CHBAL,double,CASH & DUE FROM DEPOSITORY INST,False
7,financials,SC,double,SECURITIES,False
8,financials,LNLSGR,double,LOANS AND LEASES-TOTAL,False
9,financials,LNATRES,double,ALLOW FOR LOANS LOSS ADJUSTED,False


Full definitions and source hashes are stored under metadata/selected_fields.json.


## 4. Verify failure detection before acquisition
The regression suite checks the 170-quarter scope, numeric parsing, duplicate keys, incomplete pagination, source changes, resume, corruption recovery, a completely absent manifest quarter and CSV/Parquet equivalence. Small constructed records are used only for software testing; they are never presented as FDIC findings.

In [4]:
test_run = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
                          cwd=REPO_ROOT, text=True, capture_output=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()

DOWNLOADED financials/20230331: 1
DOWNLOADED financials/20230630: 1
DOWNLOADED failures/all: 1
DOWNLOADED institutions/all: 1
VERIFIED RESUME financials/20230331: 1
VERIFIED RESUME financials/20230630: 1
VERIFIED RESUME failures/all: 1
VERIFIED RESUME institutions/all: 1
VERIFIED RESUME financials/20230630: 1
VERIFIED RESUME failures/all: 1
VERIFIED RESUME institutions/all: 1
DOWNLOADED financials/20230331: 1

test_failure_missing_cert_is_preserved (test_part1.Tests.test_failure_missing_cert_is_preserved) ... ok
test_invalid_types_and_duplicate_keys (test_part1.Tests.test_invalid_types_and_duplicate_keys) ... ok
test_pagination_empty_page_is_failure (test_part1.Tests.test_pagination_empty_page_is_failure) ... ok
test_pagination_source_change_is_failure (test_part1.Tests.test_pagination_source_change_is_failure) ... ok
test_resume_corruption_missing_quarter_and_storage (test_part1.Tests.test_resume_corruption_missing_quarter_and_storage) ... ok
test_scope (test_part1.Tests.test_scope) .

## 5. Smoke-test four quarters in an isolated directory
Download all banks for 2023Q1–Q4 and the complete auxiliary tables. This validates the live API path and restart mechanism. It is a preparatory check, not the required full dataset.

In [5]:
smoke_audit = acquire(SMOKE_ROOT, '2023-03-31', '2023-12-31')
assert smoke_audit['passed'] and smoke_audit['verified_quarters'] == 4
print('Smoke scope:', smoke_audit['start'], smoke_audit['end'])
print('Smoke rows:', smoke_audit['rows'])

VERIFIED RESUME financials/20230331: 4,740


VERIFIED RESUME financials/20230630: 4,714


VERIFIED RESUME financials/20230930: 4,686


VERIFIED RESUME financials/20231231: 4,658


VERIFIED RESUME failures/all: 4,117


VERIFIED RESUME institutions/all: 27,835


Smoke scope: 2023-03-31 2023-12-31
Smoke rows: {'financials': 18798, 'failures': 4117, 'institutions': 27835}


## 6. Acquire the complete examination scope
Do not interrupt a healthy running cell. If the runtime disconnects, rerun setup and this cell using the same Drive directory. A checkpoint is skipped only after checksum, schema, row-count and key verification. Files are replaced atomically before their manifest entry is committed. API totals and index metadata must remain stable within each paginated unit.

The source can be revised. Resume preserves verified checkpoints; a new directory is required for an entirely fresh snapshot. There is no claim that historical publication vintages have been reconstructed.

In [6]:
full_audit = acquire(FULL_ROOT, START, END)
assert full_audit['passed'] and full_audit['full_course_scope']
print('Full download and audit completed.')

VERIFIED RESUME financials/19840331: 17,930


VERIFIED RESUME financials/19840630: 17,935


VERIFIED RESUME financials/19840930: 17,946


VERIFIED RESUME financials/19841231: 17,949


VERIFIED RESUME financials/19850331: 17,968


VERIFIED RESUME financials/19850630: 17,853


... (163 outputs left out)

VERIFIED RESUME financials/20260630: 4,313


VERIFIED RESUME failures/all: 4,117


VERIFIED RESUME institutions/all: 27,835


Full download and audit completed.


## 7. Independently verify completeness and preserve provenance
The expected quarter list is constructed from the course interval, independently of the manifest. Each quarter must match its API total, contain only its reporting date, have a valid schema and unique bank–quarter keys, and match the recorded file hash. Auxiliary tables must also match their API counts and declared keys. Unexpected Parquet files fail the audit so accidental scope mixing is visible.

These checks establish acquisition integrity relative to the retrieved source. They do not establish economic correctness, absence of historical revisions or absence of all data-quality issues.

In [7]:
full_audit = audit(FULL_ROOT, START, END)
assert full_audit['expected_quarters'] == full_audit['verified_quarters'] == 170
assert full_audit['passed'] and full_audit['full_course_scope']
display(pd.DataFrame([{'table': key, 'rows': value} for key, value in full_audit['rows'].items()]))
quarter_counts = pd.DataFrame(full_audit['quarter_rows'])
display(quarter_counts)
print('Distinct financial institutions:', full_audit['distinct_financial_banks'])
print('Missing/invalid:', full_audit['missing_or_invalid'])
print('Unexpected files:', full_audit['unexpected_parquet_files'])
print('Auxiliary source observations:', full_audit['auxiliary_source_notes'])
manifest = json.loads((FULL_ROOT / 'manifest.json').read_text())
print('Recorded source indexes:', sorted({str(r.get('source_index')) for r in manifest['records'].values()}))

,table,rows
0,financials,1682616
1,failures,4117
2,institutions,27835


,quarter,rows,api_total
0,19840331,17930,17930
1,19840630,17935,17935
2,19840930,17946,17946
3,19841231,17949,17949
4,19850331,17968,17968
...,...,...,...
165,20250630,4494,4494
166,20250930,4452,4452
167,20251231,4411,4411
168,20260331,4353,4353


Distinct financial institutions: 24064
Missing/invalid: []
Unexpected files: []
Auxiliary source observations: {'failures': {'null_CERT': 488, 'resolution_counts': {'FAILURE': 3524, 'ASSISTANCE': 593}}, 'institutions': {'null_CERT': 0}}
Recorded source indexes: ["{'name': 'failures_1787667198788', 'createTimestamp': '2026-08-25T14:13:18Z'}", "{'name': 'institutions_20261113100007', 'createTimestamp': '2026-11-13T12:55:01Z'}", "{'name': 'risview_20260819185831', 'createTimestamp': '2026-08-19T18:58:33Z'}"]


## 8. Stage the verified snapshot on local disk
Copy the audited snapshot to the runtime filesystem. Its manifest, checksums and auxiliary tables are included so the benchmark can independently audit its input. The benchmark uses Financials only. This staging step is outside measured query time.

In [8]:
import shutil
LOCAL_SNAPSHOT = BENCH_ROOT / 'verified_snapshot'
shutil.copytree(FULL_ROOT, LOCAL_SNAPSHOT, dirs_exist_ok=True)
local_audit = audit(LOCAL_SNAPSHOT, START, END)
assert local_audit['passed']
print('Local snapshot verified. Free disk GiB:', round(shutil.disk_usage(BENCH_ROOT).free / 2**30, 2))

Local snapshot verified. Free disk GiB: 6.08


## 9. Compare storage layouts with identical queries
Candidates: uncompressed CSV, single-file Snappy Parquet, single-file Zstandard Parquet and one Zstandard file per year. All contain exactly the same selected Financials records. Quarterly acquisition checkpoints are also counted for storage size and file count.

Queries: (1) all rows and columns; (2) all rows, five projected columns; (3) the same five columns for reporting year 2009. Every candidate is checked against CSV before timing, with exact integer/string semantics and Float64 tolerance 1e-12. Each configuration receives an unmeasured warm-up and three measured repetitions in randomized order with seed 42. Measurements use fresh worker processes and two Arrow CPU/I/O threads.

The timer includes reading/filtering/materialization, excluding process startup, layout preparation and correctness comparisons. OS cache is not flushed. Process RSS is sampled externally every 2 ms over the worker lifetime. The sampled maximum includes imports and interpreter overhead and can miss short peaks. This is a storage benchmark using Arrow readers; Part 3 must separately compare pandas and Polars transformations.

In [9]:
from src.storage import benchmark
benchmark_result = benchmark(LOCAL_SNAPSHOT, BENCH_ROOT / 'layouts', EVIDENCE / 'storage_benchmark.json', repeats=3, year=2009)
assert benchmark_result['scope']['full_course_scope']
assert all(x['matched'] for x in benchmark_result['equivalence'])
print('Equivalence checks passed:', len(benchmark_result['equivalence']))
print('Measured runs:', len(benchmark_result['timings']))
print(benchmark_result['protocol'])

Equivalence checks passed: 9
Measured runs: 36
Same fields/rows, Arrow readers, local disk, fresh process per run, one unmeasured warm-up, randomized order. Timed region is materialized read/filter only; process startup and validation excluded. OS cache is not flushed. RSS is sampled by the parent every 2 ms over the worker lifetime, includes imports/interpreter, and may miss short peaks; it is not incremental allocation. Results do not compare pandas with Polars and do not simulate a Spark cluster.


## 10. Inspect measured tradeoffs
Review size and query-level timing separately. The same layout need not win every workload. Quarterly checkpoints support restart; yearly compaction reduces file-open overhead. Retain measured min/max values to expose timing variation. Do not interpret these warm-cache measurements as universal speedups or evidence of Spark cluster scalability.

In [10]:
sizes = pd.DataFrame(benchmark_result['sizes'])
sizes['MiB'] = sizes['bytes'] / 2**20
csv_bytes = sizes.loc[sizes['layout'].eq('csv'), 'bytes'].iloc[0]
sizes['csv_size_divided_by_layout_size'] = csv_bytes / sizes['bytes']
display(sizes)
summary = pd.DataFrame(benchmark_result['summary'])
csv_times = summary.loc[summary['layout'].eq('csv')].set_index('query')['median_seconds']
summary['csv_time_divided_by_layout_time'] = summary['query'].map(csv_times) / summary['median_seconds']
display(summary)
for query, rows in summary.groupby('query', sort=False):
    fastest = rows.loc[rows['median_seconds'].idxmin()]
    print(f"{query}: lowest measured median = {fastest['layout']} ({fastest['median_seconds']:.4f} s)")
print('A ratio above 1 means the candidate is smaller/faster than CSV for that measurement.')

,layout,bytes,files,MiB,csv_size_divided_by_layout_size
0,csv,253495928,1,241.752556,1.000000
1,parquet_snappy,139295153,1,132.842210,1.819847
2,parquet_zstd,120277721,1,114.705773,2.107588
3,year_zstd,122097002,43,116.440775,2.076185
4,acquisition_checkpoints,127663250,170,121.749163,1.985661


,layout,query,median_seconds,min_seconds,max_seconds,median_sampled_peak_process_rss_mib,csv_time_divided_by_layout_time
0,csv,full_scan,0.941578,0.859753,0.959549,400.761719,1.000000
1,csv,projection,0.461321,0.434477,0.510957,217.566406,1.000000
2,csv,year_filter,0.458899,0.454981,0.536693,219.273438,1.000000
3,parquet_snappy,full_scan,0.295722,0.289501,0.318705,547.136719,3.183994
4,parquet_snappy,projection,0.099067,0.078215,0.131179,244.925781,4.656636
5,parquet_snappy,year_filter,0.009219,0.008153,0.011786,139.148438,49.777419
6,parquet_zstd,full_scan,0.524790,0.444484,0.607682,535.320312,1.794201
7,parquet_zstd,projection,0.080416,0.075967,0.105567,238.222656,5.736644
8,parquet_zstd,year_filter,0.013223,0.010839,0.013706,139.121094,34.704790
9,year_zstd,full_scan,0.471019,0.372307,0.748244,430.410156,1.999025


full_scan: lowest measured median = parquet_snappy (0.2957 s)
projection: lowest measured median = parquet_zstd (0.0804 s)
year_filter: lowest measured median = year_zstd (0.0076 s)
A ratio above 1 means the candidate is smaller/faster than CSV for that measurement.


## 11. Save submission evidence and the executed notebook
Export aggregate results, provenance and software verification; source records are excluded. In Colab, the evidence ZIP contains only the `evidence/` directory. Extract and upload that directory to GitHub. Separately save this executed notebook using **File → Save a copy in GitHub**, with outputs enabled, at its existing repository path. Saving files in the cloned directory does not automatically save the active notebook's outputs.

Do not commit any data directory, benchmark layout or ZIP. No Part 1 report is generated here.

In [11]:
from datetime import datetime, timezone
import zipfile
save_json(EVIDENCE / 'audit.json', full_audit)
save_json(EVIDENCE / 'acquisition_manifest.json', manifest)
save_json(EVIDENCE / 'selected_fields.json', definitions)
verification = {
    'executed_at_utc': datetime.now(timezone.utc).isoformat(),
    'environment': 'Google Colab' if IN_COLAB else 'Managed Linux verification runtime',
    'python': platform.python_version(), 'platform': platform.platform(), 'versions': versions,
    'tests_exit_code': test_run.returncode, 'tests_log': test_run.stdout + test_run.stderr,
    'full_scope_passed': full_audit['passed'] and full_audit['full_course_scope'],
    'benchmark_equivalence_passed': all(x['matched'] for x in benchmark_result['equivalence']),
    'source_hashes': {str(p.relative_to(REPO_ROOT)): digest(p) for p in sorted((REPO_ROOT / 'src').glob('*.py'))},
    'limitations': 'Execution validates this source snapshot and environment. Historical revisions and publication lags require later analysis.'
}
save_json(EVIDENCE / 'verification.json', verification)
print('Evidence files:', [p.name for p in sorted(EVIDENCE.glob('*.json'))])
if IN_COLAB:
    from google.colab import files
    export_path = Path('/content/part1_evidence.zip')
    with zipfile.ZipFile(export_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for path in sorted(EVIDENCE.glob('*.json')):
            z.write(path, 'evidence/' + path.name)
    files.download(str(export_path))

Evidence files: ['acquisition_manifest.json', 'audit.json', 'selected_fields.json', 'storage_benchmark.json', 'verification.json']


## 12. Handoff to the data-quality stage
Part 1 hands over: audited source Parquet files, explicit schemas, official definitions, per-unit provenance/checksums and measured storage tradeoffs. Part 2 must investigate missingness, outliers, units, financial reporting conventions, date parsing, join coverage and exit/event semantics. Preserve source snapshots and create a separate clean layer.

The Failures endpoint's raw row count must not be used as a count of model-eligible failures without checking RESTYPE and the label definition. Current institution status and future event costs must not enter historical features. Acquisition covers 2026Q2; modelling remains restricted to observations through 2025Q2 by the examination.

**Before committing:** all 170 quarters verified; complete auxiliary tables; nine equivalence checks passed; 36 measured trials present; environment recorded; executed notebook outputs saved; no bank-level data staged. Review findings in your own words when the report is written later.

**Attribution:** supplied BFT65B examination and starter script; FDIC BankFind API and dictionaries; Apache Arrow/pandas interfaces; OpenAI ChatGPT/Codex assistance. Full references and assistance disclosure are in README.